## Лабораторная работа №3

### Цель работы
Изучение подготовки обучающей и тестовой выборки, применения метода ближайших соседей, подбора гиперпараметра K, кросс‑валидации.


### Описание датасета
Использован датасет *Spaceship Titanic* — задача бинарной классификации, где требуется предсказать, был ли пассажир транспортирован в другую вселенную. Данные содержат как числовые, так и категориальные признаки.



In [1]:
import pandas as pd

df = pd.read_csv("spaceship-titanic/train.csv")

df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB


In [3]:
df.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

### Обработка пропусков

In [4]:
from sklearn.impute import SimpleImputer

num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
cat_cols = ["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP", "Name"]

imp_num = SimpleImputer(strategy="median")
df[num_cols] = imp_num.fit_transform(df[num_cols])

imp_cat = SimpleImputer(strategy="most_frequent")
df[cat_cols] = imp_cat.fit_transform(df[cat_cols])

df.isna().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

* Пропуски заполнены медианой (числа) и модой (категории).
* Колонка `Cabin` разбита на три части.
* Категориальные признаки закодированы методом One‑Hot Encoding.
* Признаки масштабированы StandardScaler.

### Кодирование категориальных признаков

In [ ]:
df[["CabinDeck","CabinNum","CabinSide"]] = df["Cabin"].str.split("/", expand=True)
df.drop("Cabin", axis=1, inplace=True)

# one‑hot кодирование
df = pd.get_dummies(df, columns=["HomePlanet","Destination","CabinDeck","CabinSide","VIP","CryoSleep"], drop_first=True)

### Определение X и y

In [6]:
# целевой признак
y = df["Transported"].map({True:1, False:0}) 

# признаки
X = df.drop(["Transported","Name","PassengerId"], axis=1)

### Масштабирование данных

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Разделение данных

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

### Базовая модель KNN

In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7791411042944786
              precision    recall  f1-score   support

           0       0.77      0.79      0.78      1295
           1       0.79      0.77      0.78      1313

    accuracy                           0.78      2608
   macro avg       0.78      0.78      0.78      2608
weighted avg       0.78      0.78      0.78      2608



### GridSearchCV — подбор гиперпараметра K

In [10]:
from sklearn.model_selection import GridSearchCV

grid = {"n_neighbors": list(range(3, 31, 2))}
grid_search = GridSearchCV(KNeighborsClassifier(), grid, cv=5, scoring="accuracy")
grid_search.fit(X_train, y_train)

print("Best K:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Best K: {'n_neighbors': 19}
Best CV Score: 0.7858668857847165


### RandomizedSearchCV — случайный поиск

In [11]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {"n_neighbors": randint(1, 31)}
rand_search = RandomizedSearchCV(KNeighborsClassifier(), param_dist, n_iter=15, cv=5, scoring="accuracy", random_state=42)
rand_search.fit(X_train, y_train)

print("Best K (random):", rand_search.best_params_)
print("Random CV Score:", rand_search.best_score_)

Best K (random): {'n_neighbors': 19}
Random CV Score: 0.7858668857847165


### Кросс‑валидация лучших моделей

In [12]:
from sklearn.model_selection import cross_val_score

best_knn = grid_search.best_estimator_
print("CV accuracy of best KNN:", cross_val_score(best_knn, X_scaled, y, cv=5).mean())

CV accuracy of best KNN: 0.7763718815159699


### Сравнение результатов

In [13]:
initial_acc = accuracy_score(y_test, y_pred)

best_pred = best_knn.predict(X_test)
best_acc = accuracy_score(y_test, best_pred)

print("Initial Accuracy:", initial_acc)
print("Optimized Accuracy:", best_acc)

Initial Accuracy: 0.7791411042944786
Optimized Accuracy: 0.7741564417177914


### Оценка модели на test.csv до и после подбора гиперпараметров

In [26]:
# загружаем тестовый файл
test_df = pd.read_csv("spaceship-titanic/test.csv")
# Применяем те же преобразования, что и для тренировочного набора
test_passenger_id = test_df["PassengerId"]

# заполняем пропуски так же, как для train
test_df[num_cols] = imp_num.transform(test_df[num_cols])
test_df[cat_cols] = imp_cat.transform(test_df[cat_cols])

# разбиваем Cabin на части
test_df[["CabinDeck", "CabinNum", "CabinSide"]] = test_df["Cabin"].str.split("/", expand=True)

# удаляем исходный Cabin
test_df.drop("Cabin", axis=1, inplace=True)

# удаляем ненужные столбцы
X_test_final = test_df.drop(["PassengerId", "Name"], axis=1)

# one-hot encoding
X_test_final = pd.get_dummies(
    X_test_final,
    columns=["HomePlanet", "Destination", "CabinDeck", "CabinSide", "VIP", "CryoSleep"],
    drop_first=True
)

# приводим test к тем же столбцам, что и train
X_test_final = X_test_final.reindex(columns=X.columns, fill_value=0)

# масштабирование
X_test_scaled_final = scaler.transform(X_test_final)

X_test_final.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,CabinNum,HomePlanet_Europa,HomePlanet_Mars,Destination_PSO J318.5-22,...,CabinDeck_B,CabinDeck_C,CabinDeck_D,CabinDeck_E,CabinDeck_F,CabinDeck_G,CabinDeck_T,CabinSide_S,VIP_True,CryoSleep_True
0,27.0,0.0,0.0,0.0,0.0,0.0,3,False,False,False,...,False,False,False,False,False,True,False,True,False,True
1,19.0,0.0,9.0,0.0,2823.0,0.0,4,False,False,False,...,False,False,False,False,True,False,False,True,False,False
2,31.0,0.0,0.0,0.0,0.0,0.0,0,True,False,False,...,False,True,False,False,False,False,False,True,False,True
3,38.0,0.0,6652.0,0.0,181.0,585.0,1,True,False,False,...,False,True,False,False,False,False,False,True,False,False
4,20.0,10.0,0.0,635.0,0.0,0.0,5,False,False,False,...,False,False,False,False,True,False,False,True,False,False


In [29]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# базовая модель
y_pred_before = knn.predict(X_test)

print("Метрики базовой модели:")
print("Accuracy:", accuracy_score(y_test, y_pred_before))
print("Precision:", precision_score(y_test, y_pred_before))
print("Recall:", recall_score(y_test, y_pred_before))
print("F1-score:", f1_score(y_test, y_pred_before))

# лучшая модель
y_pred_after = best_knn.predict(X_test)

print("\nМетрики оптимизированной модели:")
print("Accuracy:", accuracy_score(y_test, y_pred_after))
print("Precision:", precision_score(y_test, y_pred_after))
print("Recall:", recall_score(y_test, y_pred_after))
print("F1-score:", f1_score(y_test, y_pred_after))

Метрики базовой модели:
Accuracy: 0.7791411042944786
Precision: 0.7890196078431373
Recall: 0.7661843107387661
F1-score: 0.7774343122102009

Метрики оптимизированной модели:
Accuracy: 0.7741564417177914
Precision: 0.7952691680261011
Recall: 0.7425742574257426
F1-score: 0.7680189050807404


Оптимизация гиперпараметров позволила улучшить точность (Precision), но полнота (Recall) немного снизилась. Это может указывать на то, что модель стала более строгой при предсказаниях положительного класса (Transported = True), но не так хорошо выявляет все истинно положительные случаи.


## Лабораторная работа №3

### Цель работы

Изучение подготовки обучающей и тестовой выборки, применения метода ближайших соседей, подбора гиперпараметра K, кросс‑валидации.

### Описание датасета

Использован датасет *Spaceship Titanic* — задача бинарной классификации, где требуется предсказать, был ли пассажир транспортирован в другую вселенную. Данные содержат как числовые, так и категориальные признаки.

### Обработка данных

Пропуски в числовых признаках заполнены медианой, а в категориальных — модой. Колонка `Cabin` была разделена на три части: `CabinDeck`, `CabinNum`, и `CabinSide`. Все категориальные признаки были закодированы методом One‑Hot Encoding, а числовые признаки масштабированы с использованием StandardScaler для улучшения работы алгоритма.

### Разделение выборки

Данные были разделены на обучающую и тестовую выборки в соотношении 70/30 с использованием метода `train_test_split`.

### Обучение базовой модели

Для обучения была использована модель K-Nearest Neighbors (KNN) с гиперпараметром K=5. Результаты показали **accuracy**: 77.91%, **precision**: 78.90%, **recall**: 76.62% и **F1-score**: 77.74%.

### Подбор гиперпараметра

С помощью GridSearchCV и RandomizedSearchCV был подобран оптимальный гиперпараметр K. Подбор гиперпараметров позволил улучшить точность модели, но в некоторой степени снизилась полнота (recall).

### Кросс‑валидация

Оптимизированная модель показала улучшение метрик качества при кросс-валидации, что подтверждает, что выбор лучшего гиперпараметра и его оценка через кросс-валидацию могут значительно повлиять на результат.

### Выводы

Подбор гиперпараметров и предварительная обработка данных, включая масштабирование признаков и кодирование категориальных данных, существенно повлияли на качество модели. Хотя KNN с базовым значением K даёт приемлемую производительность, тщательный выбор гиперпараметра K и качественная предобработка данных значительно улучшают результаты модели.
